In [1]:
import pandas as pd
import numpy as np
import sklearn

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.9.1


In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import sklearn

# Localizar la raíz del repositorio
ROOT = Path.cwd()

if not (ROOT / "dataset/raw/support2.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "dataset/raw/support2.csv"

print("Directorio del proyecto:", ROOT)
print("Dataset encontrado:", DATA_PATH.exists())

print("\nVersiones:")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

assert DATA_PATH.exists(), "No se encontró support2.csv"


Directorio del proyecto: C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml
Dataset encontrado: True

Versiones:
pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.9.1


In [3]:

# Cargar el dataset original SUPPORT
df = pd.read_csv(DATA_PATH)

# Verificar dimensiones
print("Dimensiones del dataset:", df.shape)
print("Número de pacientes:", df.shape[0])
print("Número de variables:", df.shape[1])

# Visualizar las primeras cinco observaciones
display(df.head())

# Verificar nombres de columnas
print("\nColumnas del dataset:")
print(df.columns.tolist())


Dimensiones del dataset: (9105, 47)
Número de pacientes: 9105
Número de variables: 47


,age,death,sex,hospdead,slos,d.time,dzgroup,dzclass,num.co,edu,...,crea,sod,ph,glucose,bun,urine,adlp,adls,sfdm2,adlsc
1,62.84998,0,male,0,5,2029,Lung Cancer,Cancer,0,11.0,...,1.199951,141.0,7.459961,NaN,NaN,NaN,7.0,7.0,NaN,7.0
2,60.33899,1,female,1,4,4,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,...,5.500000,132.0,7.250000,NaN,NaN,NaN,NaN,1.0,<2 mo. follow-up,1.0
3,52.74698,1,female,0,17,47,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,...,2.000000,134.0,7.459961,NaN,NaN,NaN,1.0,0.0,<2 mo. follow-up,0.0
4,42.38498,1,female,0,3,133,Lung Cancer,Cancer,2,11.0,...,0.799927,139.0,NaN,NaN,NaN,NaN,0.0,0.0,no(M2 and SIP pres),0.0
5,79.88495,0,female,0,16,2029,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,...,0.799927,143.0,7.509766,NaN,NaN,NaN,NaN,2.0,no(M2 and SIP pres),2.0



Columnas del dataset:
['age', 'death', 'sex', 'hospdead', 'slos', 'd.time', 'dzgroup', 'dzclass', 'num.co', 'edu', 'income', 'scoma', 'charges', 'totcst', 'totmcst', 'avtisst', 'race', 'sps', 'aps', 'surv2m', 'surv6m', 'hday', 'diabetes', 'dementia', 'ca', 'prg2m', 'prg6m', 'dnr', 'dnrday', 'meanbp', 'wblc', 'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose', 'bun', 'urine', 'adlp', 'adls', 'sfdm2', 'adlsc']


In [4]:

# 1. Tipos de datos
print("=== TIPOS DE DATOS ===")
print(df.dtypes.value_counts())

# 2. Valores faltantes por variable
missing_summary = pd.DataFrame({
    "Tipo": df.dtypes.astype(str),
    "Faltantes": df.isna().sum(),
    "Porcentaje (%)": df.isna().mean() * 100
})

missing_summary = (
    missing_summary
    .sort_values("Porcentaje (%)", ascending=False)
)

print("\n=== VARIABLES CON MÁS VALORES FALTANTES ===")
display(missing_summary.head(15).round(2))

# 3. Duplicados exactos
print("\n=== DUPLICADOS ===")
print("Filas duplicadas:", df.duplicated().sum())

# 4. Variable objetivo
print("\n=== VARIABLE OBJETIVO ===")
print("Valores faltantes:", df["hospdead"].isna().sum())

print("\nDistribución de clases:")
print(df["hospdead"].value_counts())

print("\nDistribución porcentual:")
print(
    df["hospdead"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


=== TIPOS DE DATOS ===
float64    31
int64       8
object      8
Name: count, dtype: int64

=== VARIABLES CON MÁS VALORES FALTANTES ===


,Tipo,Faltantes,Porcentaje (%)
adlp,float64,5641,61.95
urine,float64,4862,53.40
glucose,float64,4500,49.42
bun,float64,4352,47.80
totmcst,float64,3475,38.17
alb,float64,3372,37.03
income,object,2982,32.75
adls,float64,2867,31.49
bili,float64,2601,28.57
pafi,float64,2325,25.54



=== DUPLICADOS ===
Filas duplicadas: 0

=== VARIABLE OBJETIVO ===
Valores faltantes: 0

Distribución de clases:
hospdead
0    6745
1    2360
Name: count, dtype: int64

Distribución porcentual:
hospdead
0    74.08
1    25.92
Name: proportion, dtype: float64


Nos permite distinguir las variables numéricas de las categóricas.

Esto será necesario para definir las transformaciones posteriores:
- Tipo de variable:	Transformación propuesta
- Numérica:	Imputación con mediana
- Categórica:	Imputación con moda y codificación
- Numérica para modelos lineales:	Escalamiento con StandardScaler

Para prevenir data leakage, hay que identificar cuales de las variables pueden usarse como predictores porque suponiendo que querramos predecir si un paciente fallecerá durante la hospitalización. No sería correcto incluir una variable que indique cuánto duró finalmente su hospitalización (slos), porque ese valor podría conocerse solo después del desenlace y eso produciría una evaluación artificialmente optimista.

In [5]:

# Variables que requieren exclusión o revisión
candidate_exclusions = [
    "death",
    "slos",
    "d.time",
    "charges",
    "totcst",
    "totmcst",
    "avtisst",
    "sfdm2",
    "adlsc",
    "sps",
    "aps",
    "surv2m",
    "surv6m",
    "prg2m",
    "prg6m",
    "dnr",
    "dnrday",
    "hday"
]

# Construir tabla de auditoría
audit = pd.DataFrame({
    "Variable": candidate_exclusions,
    "Existe": [
        col in df.columns
        for col in candidate_exclusions
    ],
    "Tipo": [
        str(df[col].dtype) if col in df.columns else None
        for col in candidate_exclusions
    ],
    "Faltantes (%)": [
        round(df[col].isna().mean() * 100, 2)
        if col in df.columns else None
        for col in candidate_exclusions
    ]
})

display(audit)

print(
    "Variables encontradas:",
    audit["Existe"].sum(),
    "de",
    len(candidate_exclusions)
)


,Variable,Existe,Tipo,Faltantes (%)
0,death,True,int64,0.00
1,slos,True,int64,0.00
2,d.time,True,int64,0.00
3,charges,True,float64,1.89
4,totcst,True,float64,9.75
5,totmcst,True,float64,38.17
6,avtisst,True,float64,0.90
7,sfdm2,True,object,15.38
8,adlsc,True,float64,0.00
9,sps,True,float64,0.01


Variables encontradas: 18 de 18


In [6]:

# Variable objetivo
TARGET = "hospdead"

# Variables excluidas del modelado principal
EXCLUDED_COLUMNS = [
    "death",
    "slos",
    "d.time",
    "charges",
    "totcst",
    "totmcst",
    "avtisst",
    "sfdm2",
    "adlsc",
    "sps",
    "aps",
    "surv2m",
    "surv6m",
    "prg2m",
    "prg6m",
    "dnr",
    "dnrday"
]

# Construir matriz de características y objetivo
X = df.drop(columns=EXCLUDED_COLUMNS + [TARGET]).copy()
y = df[TARGET].astype(int).copy()

# Comprobaciones
print("=== DIMENSIONES ===")
print("X:", X.shape)
print("y:", y.shape)

print("\n=== VARIABLES CONSERVADAS ===")
print(X.columns.tolist())

print("\n=== VERIFICACIONES ===")
print("Objetivo excluido de X:", TARGET not in X.columns)
print("Variables excluidas:", len(EXCLUDED_COLUMNS))
print("Valores faltantes en y:", y.isna().sum())

assert TARGET not in X.columns
assert not set(EXCLUDED_COLUMNS).intersection(X.columns)
assert len(X) == len(y)

print("\nVerificaciones completadas.")


=== DIMENSIONES ===
X: (9105, 29)
y: (9105,)

=== VARIABLES CONSERVADAS ===
['age', 'sex', 'dzgroup', 'dzclass', 'num.co', 'edu', 'income', 'scoma', 'race', 'hday', 'diabetes', 'dementia', 'ca', 'meanbp', 'wblc', 'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose', 'bun', 'urine', 'adlp', 'adls']

=== VERIFICACIONES ===
Objetivo excluido de X: True
Variables excluidas: 17
Valores faltantes en y: 0

Verificaciones completadas.


\begin{cases}
0 & \text{si el paciente sobrevivió}\\
1 & \text{si el paciente falleció}
\end{cases}

TARGET = "hospdead"

Dividir los datos en entrenamiento y prueba
- Ahora aplicaremos uno de los conceptos fundamentales de la clase: un modelo debe evaluarse utilizando observaciones que no haya utilizado para aprender.

¿Por qué dividir antes de imputar?
- Por ejemplo, si calculamos la mediana de glucose utilizando los 9,105 pacientes y después dividimos el dataset. Aunque no hayamos utilizado directamente hospdead, la imputación ya habría incorporado información estadística de los pacientes reservados para prueba. Eso constituye una forma de data leakage.

Con stratify=y, buscamos que ambas particiones conserven aproximadamente esas proporciones.

Utilizaremos una partición inicial de 80% para entrenamiento y 20% para prueba, con random_state=42 para que todas las integrantes del equipo puedan reproducirla.

Esta partición aleatoria será nuestro protocolo inicial. Si al revisar el estudio encontramos una necesidad de separar pacientes por hospital o periodo, ajustaremos el diseño antes de entrenar los modelos definitivos.

In [7]:

from sklearn.model_selection import train_test_split

# División estratificada: 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Verificar dimensiones
print("=== DIMENSIONES ===")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Verificar distribución de clases
print("\n=== DISTRIBUCIÓN DE CLASES ===")

distribution = pd.DataFrame({
    "Dataset original": y.value_counts(normalize=True),
    "Entrenamiento": y_train.value_counts(normalize=True),
    "Prueba": y_test.value_counts(normalize=True)
}) * 100

display(distribution.round(2))

# Verificar que no existen índices compartidos
overlap = set(X_train.index).intersection(X_test.index)

print("\n=== VERIFICACIONES ===")
print("Índices compartidos:", len(overlap))
print("Total de observaciones:",
      len(X_train) + len(X_test))

assert len(overlap) == 0
assert len(X_train) + len(X_test) == len(X)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

print("Partición completada correctamente.")


=== DIMENSIONES ===
X_train: (7284, 29)
X_test: (1821, 29)
y_train: (7284,)
y_test: (1821,)

=== DISTRIBUCIÓN DE CLASES ===


,Dataset original,Entrenamiento,Prueba
hospdead,,,
0,74.08,74.08,74.08
1,25.92,25.92,25.92



=== VERIFICACIONES ===
Índices compartidos: 0
Total de observaciones: 9105
Partición completada correctamente.


Calcular valores faltantes en X_train
- Agrega una nueva celda debajo de la anterior.

In [8]:

# Calcular valores faltantes únicamente en entrenamiento
missing_train = pd.DataFrame({
    "Tipo": X_train.dtypes.astype(str),
    "Faltantes": X_train.isna().sum(),
    "Porcentaje (%)": X_train.isna().mean() * 100
})

# Ordenar de mayor a menor porcentaje
missing_train = missing_train.sort_values(
    "Porcentaje (%)",
    ascending=False
)

print("=== VALORES FALTANTES EN ENTRENAMIENTO ===")
display(missing_train.round(2))

# Identificar variables que superan el 50%
high_missing_cols = missing_train[
    missing_train["Porcentaje (%)"] > 50
].index.tolist()

print("\n=== VARIABLES CON MÁS DE 50% DE FALTANTES ===")
print(high_missing_cols)

print("\nCantidad de variables:", len(high_missing_cols))

# Comprobación: no modificamos las particiones
print("\nDimensiones actuales:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


=== VALORES FALTANTES EN ENTRENAMIENTO ===


,Tipo,Faltantes,Porcentaje (%)
adlp,float64,4505,61.85
urine,float64,3897,53.50
glucose,float64,3610,49.56
bun,float64,3489,47.90
alb,float64,2686,36.88
income,object,2366,32.48
adls,float64,2291,31.45
bili,float64,2116,29.05
pafi,float64,1869,25.66
ph,float64,1841,25.27



=== VARIABLES CON MÁS DE 50% DE FALTANTES ===
['adlp', 'urine']

Cantidad de variables: 2

Dimensiones actuales:
X_train: (7284, 29)
X_test: (1821, 29)


Eliminar variables con más del 50% de faltantes

In [9]:

# Eliminar las mismas columnas de entrenamiento y prueba
X_train = X_train.drop(columns=high_missing_cols).copy()
X_test = X_test.drop(columns=high_missing_cols).copy()

print("=== VARIABLES ELIMINADAS ===")
print(high_missing_cols)

print("\n=== DIMENSIONES ACTUALIZADAS ===")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\n=== VERIFICACIONES ===")

# Ambos conjuntos deben tener las mismas columnas
assert X_train.columns.equals(X_test.columns)

# Las columnas eliminadas no deben estar presentes
assert not set(high_missing_cols).intersection(X_train.columns)
assert not set(high_missing_cols).intersection(X_test.columns)

# Mantener el número de observaciones
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

print("Columnas consistentes:", X_train.columns.equals(X_test.columns))
print("Número de características:", X_train.shape[1])
print("Eliminación completada correctamente.")


=== VARIABLES ELIMINADAS ===
['adlp', 'urine']

=== DIMENSIONES ACTUALIZADAS ===
X_train: (7284, 27)
X_test: (1821, 27)

=== VERIFICACIONES ===
Columnas consistentes: True
Número de características: 27
Eliminación completada correctamente.


Se estableció un umbral de exclusión del 50% de valores faltantes, calculado exclusivamente sobre el conjunto de entrenamiento. Las variables adlp y urine superaron este criterio, con porcentajes de ausencia del 61.85% y 53.50%, respectivamente, por lo que fueron excluidas de ambos conjuntos. Las variables restantes conservaron sus valores faltantes para un tratamiento posterior mediante imputación.

Identificación de tipos de variables

In [10]:

# Identificar variables según su tipo de dato
numeric_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("=== VARIABLES NUMÉRICAS ===")
print(numeric_cols)
print("Cantidad:", len(numeric_cols))

print("\n=== VARIABLES CATEGÓRICAS ===")
print(categorical_cols)
print("Cantidad:", len(categorical_cols))

print("\n=== VERIFICACIONES ===")

# Todas las columnas deben estar clasificadas
assert len(numeric_cols) + len(categorical_cols) == X_train.shape[1]

# No debe haber columnas repetidas entre grupos
assert set(numeric_cols).isdisjoint(set(categorical_cols))

print("Total de variables clasificadas:",
      len(numeric_cols) + len(categorical_cols))

print("Todas las columnas clasificadas correctamente.")


=== VARIABLES NUMÉRICAS ===
['age', 'num.co', 'edu', 'scoma', 'hday', 'diabetes', 'dementia', 'meanbp', 'wblc', 'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose', 'bun', 'adls']
Cantidad: 21

=== VARIABLES CATEGÓRICAS ===
['sex', 'dzgroup', 'dzclass', 'income', 'race', 'ca']
Cantidad: 6

=== VERIFICACIONES ===
Total de variables clasificadas: 27
Todas las columnas clasificadas correctamente.


In [11]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Pipeline para variables numéricas
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

print("=== PIPELINE NUMÉRICO ===")
print(numeric_pipeline)

print("\n=== VERIFICACIONES ===")
print("Estrategia:",
      numeric_pipeline.named_steps["imputer"].strategy)

print("Cantidad de variables numéricas:", len(numeric_cols))

assert numeric_pipeline.named_steps["imputer"].strategy == "median"

print("Pipeline numérico definido correctamente.")


=== PIPELINE NUMÉRICO ===

Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

=== VERIFICACIONES ===
Estrategia: median
Cantidad de variables numéricas: 21
Pipeline numérico definido correctamente.


In [12]:

from sklearn.preprocessing import OneHotEncoder

# Pipeline para variables categóricas
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("=== PIPELINE CATEGÓRICO ===")
print(categorical_pipeline)

print("\n=== VERIFICACIONES ===")

print(
    "Estrategia de imputación:",
    categorical_pipeline.named_steps["imputer"].strategy
)

print(
    "Tratamiento de categorías desconocidas:",
    categorical_pipeline.named_steps["encoder"].handle_unknown
)

print(
    "Cantidad de variables categóricas:",
    len(categorical_cols)
)

assert (
    categorical_pipeline.named_steps["imputer"].strategy
    == "most_frequent"
)

assert (
    categorical_pipeline.named_steps["encoder"].handle_unknown
    == "ignore"
)

print("Pipeline categórico definido correctamente.")


=== PIPELINE CATEGÓRICO ===
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])

=== VERIFICACIONES ===
Estrategia de imputación: most_frequent
Tratamiento de categorías desconocidas: ignore
Cantidad de variables categóricas: 6
Pipeline categórico definido correctamente.


In [13]:

from sklearn.compose import ColumnTransformer

# Combinar los pipelines numérico y categórico
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

print("=== PREPROCESADOR COMPLETO ===")
print(preprocessor)

print("\n=== VERIFICACIONES ===")

print("Cantidad de transformaciones:",
      len(preprocessor.transformers))

print("Columnas numéricas:",
      len(preprocessor.transformers[0][2]))

print("Columnas categóricas:",
      len(preprocessor.transformers[1][2]))

print("Tratamiento de columnas restantes:",
      preprocessor.remainder)

assert len(preprocessor.transformers) == 2
assert len(preprocessor.transformers[0][2]) == 21
assert len(preprocessor.transformers[1][2]) == 6
assert preprocessor.remainder == "drop"

print("\nColumnTransformer definido correctamente.")


=== PREPROCESADOR COMPLETO ===
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['age', 'num.co', 'edu', 'scoma', 'hday',
                                  'diabetes', 'dementia', 'meanbp', 'wblc',
                                  'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili',
                                  'crea', 'sod', 'ph', 'glucose', 'bun',
                                  'adls']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['sex', 'dzgroup', 'dzclass', 'income', '

In [14]:

from sklearn.base import clone
import numpy as np

# Crear una copia independiente para verificar el preprocesamiento
preprocessor_check = clone(preprocessor)

# Ajustar y transformar solamente el conjunto de entrenamiento
X_train_check = preprocessor_check.fit_transform(X_train)

# Obtener los nombres de las características transformadas
feature_names = preprocessor_check.get_feature_names_out()

print("=== DIMENSIONES ===")
print("X_train original:", X_train.shape)
print("X_train transformado:", X_train_check.shape)

print("\n=== VALORES FALTANTES ===")

# Convertir a array denso solo para esta comprobación
X_train_check_dense = (
    X_train_check.toarray()
    if hasattr(X_train_check, "toarray")
    else X_train_check
)

print(
    "Valores faltantes después del preprocesamiento:",
    np.isnan(X_train_check_dense).sum()
)

print("\n=== CARACTERÍSTICAS GENERADAS ===")
print("Cantidad total:", len(feature_names))

print("\nPrimeras 15 características:")
print(feature_names[:15].tolist())

print("\nÚltimas 15 características:")
print(feature_names[-15:].tolist())

print("\n=== VERIFICACIONES ===")
assert X_train_check.shape[0] == len(X_train)
assert X_train_check.shape[1] == len(feature_names)
assert np.isnan(X_train_check_dense).sum() == 0

print("Preprocesamiento verificado correctamente.")


=== DIMENSIONES ===
X_train original: (7284, 27)
X_train transformado: (7284, 47)

=== VALORES FALTANTES ===
Valores faltantes después del preprocesamiento: 0

=== CARACTERÍSTICAS GENERADAS ===
Cantidad total: 47

Primeras 15 características:
['numeric__age', 'numeric__num.co', 'numeric__edu', 'numeric__scoma', 'numeric__hday', 'numeric__diabetes', 'numeric__dementia', 'numeric__meanbp', 'numeric__wblc', 'numeric__hrt', 'numeric__resp', 'numeric__temp', 'numeric__pafi', 'numeric__alb', 'numeric__bili']

Últimas 15 características:
['categorical__dzclass_COPD/CHF/Cirrhosis', 'categorical__dzclass_Cancer', 'categorical__dzclass_Coma', 'categorical__income_$11-$25k', 'categorical__income_$25-$50k', 'categorical__income_>$50k', 'categorical__income_under $11k', 'categorical__race_asian', 'categorical__race_black', 'categorical__race_hispanic', 'categorical__race_other', 'categorical__race_white', 'categorical__ca_metastatic', 'categorical__ca_no', 'categorical__ca_yes']

=== VERIFICACIONES

In [15]:

from sklearn.preprocessing import StandardScaler

# Pipeline numérico para modelos lineales
numeric_pipeline_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocesador completo para regresión logística
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_scaled, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

print("=== PIPELINE NUMÉRICO CON ESCALAMIENTO ===")
print(numeric_pipeline_scaled)

print("\n=== VERIFICACIONES ===")

print(
    "Estrategia de imputación:",
    numeric_pipeline_scaled.named_steps["imputer"].strategy
)

print(
    "Escalador:",
    type(numeric_pipeline_scaled.named_steps["scaler"]).__name__
)

print(
    "Variables numéricas:",
    len(preprocessor_scaled.transformers[0][2])
)

print(
    "Variables categóricas:",
    len(preprocessor_scaled.transformers[1][2])
)

assert numeric_pipeline_scaled.named_steps["imputer"].strategy == "median"
assert isinstance(
    numeric_pipeline_scaled.named_steps["scaler"],
    StandardScaler
)
assert len(preprocessor_scaled.transformers[0][2]) == 21
assert len(preprocessor_scaled.transformers[1][2]) == 6

print("\nPreprocesador escalado definido correctamente.")


=== PIPELINE NUMÉRICO CON ESCALAMIENTO ===
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

=== VERIFICACIONES ===
Estrategia de imputación: median
Escalador: StandardScaler
Variables numéricas: 21
Variables categóricas: 6

Preprocesador escalado definido correctamente.


Usaremos DummyClassifier(strategy="prior") de scikit-learn.

Este clasificador:
- Aprende la distribución de clases del conjunto de entrenamiento.
- Predice siempre la clase mayoritaria.
- Produce probabilidades basadas en las proporciones observadas durante el entrenamiento.
- No aprende relaciones entre las características clínicas y el desenlace.

In [16]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Modelo baseline: predice la clase mayoritaria
baseline_model = DummyClassifier(
    strategy="prior"
)

# Entrenar solo con el conjunto de entrenamiento
baseline_model.fit(X_train, y_train)

# Predicciones sobre entrenamiento y prueba
y_train_pred_baseline = baseline_model.predict(X_train)
y_test_pred_baseline = baseline_model.predict(X_test)

print("=== BASELINE: DUMMY CLASSIFIER ===")

print("Clases:", baseline_model.classes_)

print(
    "Distribución aprendida:",
    baseline_model.class_prior_
)

print(
    "Clase predicha:",
    baseline_model.predict(X_test.iloc[:1])[0]
)

print("\n=== ACCURACY ===")

train_accuracy = accuracy_score(
    y_train,
    y_train_pred_baseline
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred_baseline
)

print(f"Accuracy entrenamiento: {train_accuracy:.4f}")
print(f"Accuracy prueba: {test_accuracy:.4f}")

print("\n=== VERIFICACIONES ===")

print(
    "Clases predichas en prueba:",
    np.unique(y_test_pred_baseline)
)

assert len(y_test_pred_baseline) == len(y_test)
assert np.all(y_test_pred_baseline == 0)

print("Baseline entrenado correctamente.")


=== BASELINE: DUMMY CLASSIFIER ===
Clases: [0 1]
Distribución aprendida: [0.74080176 0.25919824]
Clase predicha: 0

=== ACCURACY ===
Accuracy entrenamiento: 0.7408
Accuracy prueba: 0.7408

=== VERIFICACIONES ===
Clases predichas en prueba: [0]
Baseline entrenado correctamente.


In [17]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Predicciones y probabilidades sobre prueba
y_pred_baseline = baseline_model.predict(X_test)
y_proba_baseline = baseline_model.predict_proba(X_test)[:, 1]

# Calcular métricas
baseline_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_baseline),
    "Precision": precision_score(
        y_test, y_pred_baseline, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred_baseline, zero_division=0
    ),
    "F1-score": f1_score(
        y_test, y_pred_baseline, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test, y_proba_baseline
    ),
    "Average Precision": average_precision_score(
        y_test, y_proba_baseline
    )
}

print("=== MÉTRICAS DEL BASELINE ===")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")

# Matriz de confusión
cm_baseline = confusion_matrix(
    y_test,
    y_pred_baseline,
    labels=[0, 1]
)

print("\n=== MATRIZ DE CONFUSIÓN ===")
print(cm_baseline)

# Reporte de clasificación
print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_test,
        y_pred_baseline,
        labels=[0, 1],
        zero_division=0
    )
)

print("\n=== VERIFICACIONES ===")
assert cm_baseline.sum() == len(y_test)
assert baseline_metrics["Recall"] == 0
assert baseline_metrics["F1-score"] == 0

print("Evaluación del baseline completada.")


=== MÉTRICAS DEL BASELINE ===
Accuracy: 0.7408
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
ROC-AUC: 0.5000
Average Precision: 0.2592

=== MATRIZ DE CONFUSIÓN ===
[[1349    0]
 [ 472    0]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.74      1.00      0.85      1349
           1       0.00      0.00      0.00       472

    accuracy                           0.74      1821
   macro avg       0.37      0.50      0.43      1821
weighted avg       0.55      0.74      0.63      1821


=== VERIFICACIONES ===
Evaluación del baseline completada.


In [18]:

# Carpeta de resultados del proyecto
RESULTS_DIR = ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Crear tabla de métricas del baseline
baseline_results_df = pd.DataFrame([
    {
        "Modelo": "DummyClassifier_prior",
        "Accuracy": baseline_metrics["Accuracy"],
        "Precision": baseline_metrics["Precision"],
        "Recall": baseline_metrics["Recall"],
        "F1-score": baseline_metrics["F1-score"],
        "ROC-AUC": baseline_metrics["ROC-AUC"],
        "Average Precision": baseline_metrics["Average Precision"]
    }
])

# Guardar métricas
metrics_path = RESULTS_DIR / "baseline_metrics.csv"
baseline_results_df.to_csv(metrics_path, index=False)

# Guardar matriz de confusión
confusion_df = pd.DataFrame(
    cm_baseline,
    index=["Real_0", "Real_1"],
    columns=["Predicho_0", "Predicho_1"]
)

confusion_path = RESULTS_DIR / "baseline_confusion_matrix.csv"
confusion_df.to_csv(confusion_path)

# Verificar archivos
print("=== ARCHIVOS GENERADOS ===")
print("Métricas:", metrics_path)
print("Matriz de confusión:", confusion_path)

print("\n=== MÉTRICAS GUARDADAS ===")
display(baseline_results_df.round(4))

print("\n=== MATRIZ DE CONFUSIÓN GUARDADA ===")
display(confusion_df)

print("\n=== VERIFICACIONES ===")
assert metrics_path.exists()
assert confusion_path.exists()

print("Resultados del baseline guardados correctamente.")


=== ARCHIVOS GENERADOS ===


Métricas: C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\results\tables\baseline_metrics.csv
Matriz de confusión: C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\results\tables\baseline_confusion_matrix.csv

=== MÉTRICAS GUARDADAS ===


,Modelo,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision
0,DummyClassifier_prior,0.7408,0.0,0.0,0.0,0.5,0.2592



=== MATRIZ DE CONFUSIÓN GUARDADA ===


,Predicho_0,Predicho_1
Real_0,1349,0
Real_1,472,0



=== VERIFICACIONES ===
Resultados del baseline guardados correctamente.


In [19]:

# Directorio de datos procesados
PROCESSED_DIR = ROOT / "dataset" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Guardar los índices originales de cada partición
train_indices = pd.DataFrame({
    "original_index": X_train.index
})

test_indices = pd.DataFrame({
    "original_index": X_test.index
})

train_path = PROCESSED_DIR / "train_indices.csv"
test_path = PROCESSED_DIR / "test_indices.csv"

train_indices.to_csv(train_path, index=False)
test_indices.to_csv(test_path, index=False)

print("=== ARCHIVOS GENERADOS ===")
print("Entrenamiento:", train_path)
print("Prueba:", test_path)

print("\n=== VERIFICACIONES ===")

# Volver a leer los archivos
saved_train = pd.read_csv(train_path)["original_index"]
saved_test = pd.read_csv(test_path)["original_index"]

print("Índices de entrenamiento:", len(saved_train))
print("Índices de prueba:", len(saved_test))

# Comprobar que los índices guardados coinciden con los originales
assert saved_train.tolist() == X_train.index.tolist()
assert saved_test.tolist() == X_test.index.tolist()

# Comprobar que las particiones no se superponen
assert set(saved_train).isdisjoint(set(saved_test))

# Comprobar que cubren las 9105 observaciones
assert len(saved_train) + len(saved_test) == len(df)

print("Índices de entrenamiento correctos: True")
print("Índices de prueba correctos: True")
print("Sin superposición: True")
print("Partición reproducible guardada correctamente.")


=== ARCHIVOS GENERADOS ===
Entrenamiento: C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\dataset\processed\train_indices.csv
Prueba: C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\dataset\processed\test_indices.csv

=== VERIFICACIONES ===
Índices de entrenamiento: 7284
Índices de prueba: 1821
Índices de entrenamiento correctos: True
Índices de prueba correctos: True
Sin superposición: True
Partición reproducible guardada correctamente.


In [20]:

import json

# Configuración reproducible del experimento
preprocessing_config = {
    "dataset": "SUPPORT2",
    "target": TARGET,
    "random_state": 42,
    "test_size": 0.20,
    "stratified_split": True,
    "missing_threshold": 0.50,
    "initial_excluded_columns": EXCLUDED_COLUMNS,
    "high_missing_columns": high_missing_cols,
    "selected_features": X_train.columns.tolist(),
    "numeric_features": numeric_cols,
    "categorical_features": categorical_cols,
    "numeric_imputation": "median",
    "categorical_imputation": "most_frequent",
    "categorical_encoding": "onehot",
    "handle_unknown_categories": "ignore",
    "linear_model_scaling": "StandardScaler"
}

# Guardar la configuración
config_path = PROCESSED_DIR / "preprocessing_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        preprocessing_config,
        file,
        indent=4,
        ensure_ascii=False
    )

print("=== ARCHIVO GENERADO ===")
print(config_path)

# Volver a leer para verificar
with open(config_path, "r", encoding="utf-8") as file:
    saved_config = json.load(file)

print("\n=== CONFIGURACIÓN GUARDADA ===")
print("Dataset:", saved_config["dataset"])
print("Variable objetivo:", saved_config["target"])
print("Semilla:", saved_config["random_state"])
print("Proporción de prueba:", saved_config["test_size"])

print("\nVariables excluidas inicialmente:",
      len(saved_config["initial_excluded_columns"]))

print("Variables eliminadas por faltantes:",
      saved_config["high_missing_columns"])

print("Características finales:",
      len(saved_config["selected_features"]))

print("Numéricas:", len(saved_config["numeric_features"]))
print("Categóricas:", len(saved_config["categorical_features"]))

print("\n=== VERIFICACIONES ===")

assert saved_config["selected_features"] == X_train.columns.tolist()
assert saved_config["numeric_features"] == numeric_cols
assert saved_config["categorical_features"] == categorical_cols
assert saved_config["high_missing_columns"] == high_missing_cols
assert len(saved_config["selected_features"]) == 27

print("Configuración guardada correctamente.")


=== ARCHIVO GENERADO ===
C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\dataset\processed\preprocessing_config.json

=== CONFIGURACIÓN GUARDADA ===
Dataset: SUPPORT2
Variable objetivo: hospdead
Semilla: 42
Proporción de prueba: 0.2

Variables excluidas inicialmente: 17
Variables eliminadas por faltantes: ['adlp', 'urine']
Características finales: 27
Numéricas: 21
Categóricas: 6

=== VERIFICACIONES ===
Configuración guardada correctamente.


JSON permite representar listas, números y parámetros dentro de un mismo archivo.

In [21]:

# El módulo preprocessing.py (raíz del repositorio) es la fuente de verdad
# del preprocesamiento. En P1 esta celda lo generaba; en P2 el módulo se
# amplió (grupos de variables, estrategias de imputación, indicadores de
# faltantes), por lo que ya NO se sobrescribe desde este notebook: solo se
# verifica que exista y que conserve la interfaz usada en P1.
module_path = ROOT / "preprocessing.py"
assert module_path.exists(), "No se encontró preprocessing.py en la raíz"

module_source = module_path.read_text(encoding="utf-8")
for required in ("def load_config", "def load_data", "def build_preprocessor"):
    assert required in module_source, f"Falta {required} en preprocessing.py"

compile(module_source, str(module_path), "exec")

print("=== MÓDULO DE PREPROCESAMIENTO ===")
print(module_path)
print("Interfaz de P1 disponible: load_config, load_data, build_preprocessor")

=== MÓDULO DE PREPROCESAMIENTO ===
C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\preprocessing.py
Interfaz de P1 disponible: load_config, load_data, build_preprocessor


In [22]:

import sys
import importlib

# Permitir importar módulos desde la raíz del proyecto
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Importar el módulo que acabamos de crear
import preprocessing

# Recargarlo por si realizamos modificaciones
importlib.reload(preprocessing)

# Importar sus funciones
from preprocessing import (
    load_config,
    load_data,
    build_preprocessor
)

print("=== 1. VERIFICAR CONFIGURACIÓN ===")

config_check = load_config()

print("Variable objetivo:", config_check["target"])
print("Características:", len(config_check["selected_features"]))

# Recuperar las particiones desde los archivos guardados
X_train_loaded, X_test_loaded, y_train_loaded, y_test_loaded = load_data()

print("\n=== 2. VERIFICAR PARTICIONES ===")
print("X_train recuperado:", X_train_loaded.shape)
print("X_test recuperado:", X_test_loaded.shape)
print("y_train recuperado:", y_train_loaded.shape)
print("y_test recuperado:", y_test_loaded.shape)

# Comparar con las particiones originales del notebook
pd.testing.assert_frame_equal(X_train_loaded, X_train)
pd.testing.assert_frame_equal(X_test_loaded, X_test)

pd.testing.assert_series_equal(y_train_loaded, y_train)
pd.testing.assert_series_equal(y_test_loaded, y_test)

print("Particiones idénticas a las originales: True")

print("\n=== 3. VERIFICAR PREPROCESADORES ===")

prep_tree = build_preprocessor(scaled=False)
prep_linear = build_preprocessor(scaled=True)

print("Preprocesador para árboles:",
      type(prep_tree).__name__)

print("Preprocesador para modelos lineales:",
      type(prep_linear).__name__)

# Revisar los pasos del pipeline numérico
tree_steps = list(
    prep_tree.named_transformers_["numeric"].named_steps
) if hasattr(prep_tree, "transformers_") else list(
    prep_tree.transformers[0][1].named_steps
)

linear_steps = list(
    prep_linear.transformers[0][1].named_steps
)

print("Pasos numéricos para árboles:", tree_steps)
print("Pasos numéricos para modelos lineales:", linear_steps)

assert tree_steps == ["imputer"]
assert linear_steps == ["imputer", "scaler"]

print("\n=== RESULTADO ===")
print("Módulo reutilizable verificado correctamente.")


=== 1. VERIFICAR CONFIGURACIÓN ===
Variable objetivo: hospdead
Características: 27



=== 2. VERIFICAR PARTICIONES ===


X_train recuperado: (7284, 27)
X_test recuperado: (1821, 27)
y_train recuperado: (7284,)
y_test recuperado: (1821,)
Particiones idénticas a las originales: True

=== 3. VERIFICAR PREPROCESADORES ===
Preprocesador para árboles: ColumnTransformer
Preprocesador para modelos lineales: ColumnTransformer
Pasos numéricos para árboles: ['imputer']
Pasos numéricos para modelos lineales: ['imputer', 'scaler']

=== RESULTADO ===
Módulo reutilizable verificado correctamente.


In [23]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score

# Construir un pipeline de prueba
integration_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(scaled=False)),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

print("=== PIPELINE DE INTEGRACIÓN ===")
print(integration_pipeline)

# Ajustar únicamente con entrenamiento
integration_pipeline.fit(X_train_loaded, y_train_loaded)

# Verificar predicciones sobre entrenamiento
y_train_integration_pred = integration_pipeline.predict(
    X_train_loaded
)

print("\n=== RESULTADOS DE VERIFICACIÓN ===")
print("Predicciones generadas:", len(y_train_integration_pred))

print(
    "Clases predichas:",
    np.unique(y_train_integration_pred)
)

print(
    "Accuracy entrenamiento:",
    round(
        accuracy_score(
            y_train_loaded,
            y_train_integration_pred
        ),
        4
    )
)

print("\n=== VERIFICACIONES ===")

assert len(y_train_integration_pred) == len(y_train_loaded)
assert integration_pipeline.named_steps["classifier"].tree_ is not None

print("Pipeline de clasificación integrado correctamente.")


=== PIPELINE DE INTEGRACIÓN ===
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'num.co', 'edu',
                                                   'scoma', 'hday', 'diabetes',
                                                   'dementia', 'meanbp', 'wblc',
                                                   'hrt', 'resp', 'temp',
                                                   'pafi', 'alb', 'bili',
                                                   'crea', 'sod', 'ph',
                                                   'glucose', 'bun', 'adls']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
              


=== RESULTADOS DE VERIFICACIÓN ===
Predicciones generadas: 7284
Clases predichas: [0 1]
Accuracy entrenamiento: 0.7787

=== VERIFICACIONES ===
Pipeline de clasificación integrado correctamente.


In [24]:

from sklearn.model_selection import StratifiedKFold, cross_validate

# Definir validación cruzada estratificada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluar el pipeline únicamente con entrenamiento
cv_results = cross_validate(
    integration_pipeline,
    X_train_loaded,
    y_train_loaded,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    return_train_score=False,
    error_score="raise"
)

print("=== VALIDACIÓN CRUZADA: 5 FOLDS ===")

cv_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": cv_results["test_accuracy"],
    "Recall": cv_results["test_recall"],
    "F1-score": cv_results["test_f1"],
    "ROC-AUC": cv_results["test_roc_auc"]
})

display(cv_summary.round(4))

print("\n=== PROMEDIOS ===")
print(
    cv_summary.drop(columns="Fold")
    .mean()
    .round(4)
)

print("\n=== VERIFICACIONES ===")
print("Cantidad de folds:", len(cv_results["test_accuracy"]))

assert len(cv_results["test_accuracy"]) == 5
assert cv_summary.drop(columns="Fold").notna().all().all()

print("Validación cruzada ejecutada correctamente.")


=== VALIDACIÓN CRUZADA: 5 FOLDS ===


,Fold,Accuracy,Recall,F1-score,ROC-AUC
0,1,0.7735,0.3103,0.4149,0.7692
1,2,0.7708,0.2169,0.3293,0.7429
2,3,0.7721,0.1958,0.3083,0.7768
3,4,0.7783,0.2434,0.3629,0.7628
4,5,0.7706,0.1645,0.2707,0.7691



=== PROMEDIOS ===
Accuracy    0.7731
Recall      0.2262
F1-score    0.3372
ROC-AUC     0.7642
dtype: float64

=== VERIFICACIONES ===
Cantidad de folds: 5
Validación cruzada ejecutada correctamente.


In [25]:

from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

print("=== 1. INTEGRIDAD DEL CONJUNTO DE PRUEBA ===")

# Recuperar las observaciones de prueba del archivo original
saved_test_indices = pd.read_csv(
    PROCESSED_DIR / "test_indices.csv"
)["original_index"].tolist()

X_test_original = df.loc[
    saved_test_indices,
    X_test.columns
].copy()

y_test_original = df.loc[
    saved_test_indices,
    TARGET
].astype(int).copy()

# Comparar con los conjuntos actuales
pd.testing.assert_frame_equal(
    X_test,
    X_test_original
)

pd.testing.assert_series_equal(
    y_test,
    y_test_original
)

print("X_test conserva sus valores originales: True")
print("y_test conserva sus etiquetas originales: True")

print("\n=== 2. ESTADO DE LOS PREPROCESADORES ===")

# Verificar que los preprocesadores compartidos no fueron ajustados
for name, prep in [
    ("preprocessor", preprocessor),
    ("preprocessor_scaled", preprocessor_scaled)
]:
    try:
        check_is_fitted(prep)
        fitted = True
    except NotFittedError:
        fitted = False

    print(f"{name} ajustado: {fitted}")
    assert not fitted

print("\n=== RESULTADO ===")
print("Integridad de datos y preprocesadores verificada.")


=== 1. INTEGRIDAD DEL CONJUNTO DE PRUEBA ===
X_test conserva sus valores originales: True
y_test conserva sus etiquetas originales: True

=== 2. ESTADO DE LOS PREPROCESADORES ===
preprocessor ajustado: False
preprocessor_scaled ajustado: False

=== RESULTADO ===
Integridad de datos y preprocesadores verificada.


In [26]:

print("=== AUDITORÍA TEMPORAL DE SUPPORT ===")

# Estadísticas del día de ingreso al estudio
print("\nResumen de hday:")
display(df["hday"].describe())

# Frecuencia de los días de ingreso al estudio
hday_distribution = (
    df["hday"]
    .value_counts()
    .sort_index()
    .rename_axis("hday")
    .reset_index(name="Pacientes")
)

hday_distribution["Porcentaje (%)"] = (
    hday_distribution["Pacientes"] / len(df) * 100
)

print("\nPrimeros 20 días registrados:")
display(hday_distribution.head(20).round(2))

# Comprobar cuántos ingresaron al estudio después del día 1
print("\n=== RESUMEN ===")

print("Pacientes con hday = 1:",
      (df["hday"] == 1).sum())

print("Pacientes con hday > 1:",
      (df["hday"] > 1).sum())

print("Porcentaje con hday > 1:",
      round((df["hday"] > 1).mean() * 100, 2))

print("Valores faltantes en hday:",
      df["hday"].isna().sum())

print("Auditoría descriptiva completada.")


=== AUDITORÍA TEMPORAL DE SUPPORT ===

Resumen de hday:


count    9105.000000
mean        4.400000
std         9.126811
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max       148.000000
Name: hday, dtype: float64


Primeros 20 días registrados:


,hday,Pacientes,Porcentaje (%)
0,1,5948,65.33
1,2,496,5.45
2,3,398,4.37
3,4,315,3.46
4,5,218,2.39
5,6,187,2.05
6,7,170,1.87
7,8,148,1.63
8,9,107,1.18
9,10,105,1.15



=== RESUMEN ===
Pacientes con hday = 1: 5948
Pacientes con hday > 1: 3157
Porcentaje con hday > 1: 34.67
Valores faltantes en hday: 0
Auditoría descriptiva completada.


In [27]:

print("=== DURACIÓN DESDE INGRESO AL ESTUDIO ===")

# Revisar duración hasta el alta
display(df["slos"].describe())

# Conteos según duración
print("\n=== POSIBLES ALTAS TEMPRANAS ===")

print("slos <= 0:", (df["slos"] <= 0).sum())
print("slos <= 1:", (df["slos"] <= 1).sum())
print("slos <= 2:", (df["slos"] <= 2).sum())
print("slos <= 3:", (df["slos"] <= 3).sum())

# Distribución de mortalidad en los grupos
audit_early_discharge = pd.crosstab(
    df["slos"] <= 2,
    df["hospdead"],
    margins=True
)

print("\n=== MORTALIDAD SEGÚN slos <= 2 ===")
display(audit_early_discharge)

print("\n=== VERIFICACIONES ===")
print("Pacientes totales:", len(df))
print("Valores faltantes slos:", df["slos"].isna().sum())

assert audit_early_discharge.loc["All", "All"] == len(df)

print("Auditoría completada correctamente.")


=== DURACIÓN DESDE INGRESO AL ESTUDIO ===


count    9105.000000
mean       17.863042
std        22.006440
min         3.000000
25%         6.000000
50%        11.000000
75%        20.000000
max       343.000000
Name: slos, dtype: float64


=== POSIBLES ALTAS TEMPRANAS ===
slos <= 0: 0
slos <= 1: 0
slos <= 2: 0
slos <= 3: 347

=== MORTALIDAD SEGÚN slos <= 2 ===


hospdead,0,1,All
slos,,,
False,6745,2360,9105
All,6745,2360,9105



=== VERIFICACIONES ===
Pacientes totales: 9105
Valores faltantes slos: 0
Auditoría completada correctamente.


In [28]:

# Identificar pacientes con duración de 3 días
day3_patients = df[df["slos"] == 3].copy()

print("=== PACIENTES CON slos = 3 ===")
print("Cantidad:", len(day3_patients))

# Distribución de mortalidad
day3_mortality = (
    day3_patients["hospdead"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

print("\n=== DISTRIBUCIÓN DE MORTALIDAD ===")
print(day3_mortality)

print("\n=== PORCENTAJES ===")
print(
    (day3_mortality / len(day3_patients) * 100).round(2)
)

# Comparar mortalidad según duración
comparison = pd.crosstab(
    df["slos"] == 3,
    df["hospdead"],
    margins=True
)

print("\n=== COMPARACIÓN CON EL RESTO ===")
display(comparison)

print("\n=== VERIFICACIONES ===")
assert len(day3_patients) == 347
assert day3_mortality.sum() == 347

print("Auditoría del grupo completada.")


=== PACIENTES CON slos = 3 ===
Cantidad: 347

=== DISTRIBUCIÓN DE MORTALIDAD ===
hospdead
0    209
1    138
Name: count, dtype: int64

=== PORCENTAJES ===
hospdead
0    60.23
1    39.77
Name: count, dtype: float64

=== COMPARACIÓN CON EL RESTO ===


hospdead,0,1,All
slos,,,
False,6536,2222,8758
True,209,138,347
All,6745,2360,9105



=== VERIFICACIONES ===
Auditoría del grupo completada.


In [29]:

print("=== AUDITORÍA DE TIEMPO DE FALLECIMIENTO ===")

# Pacientes fallecidos durante la hospitalización
hospital_deaths = df[df["hospdead"] == 1].copy()

print("Fallecidos hospitalarios:", len(hospital_deaths))

print("\n=== RESUMEN DE d.time ===")
display(hospital_deaths["d.time"].describe())

# Identificar fallecimientos hospitalarios con seguimiento corto
print("\n=== TIEMPOS DE SEGUIMIENTO ===")

for days in [1, 2, 3]:
    count = (
        hospital_deaths["d.time"] <= days
    ).sum()

    print(f"d.time <= {days}: {count}")

# Revisar los fallecidos con slos = 3
day3_deaths = df[
    (df["slos"] == 3) &
    (df["hospdead"] == 1)
].copy()

print("\n=== FALLECIDOS CON slos = 3 ===")
print("Cantidad:", len(day3_deaths))

print("\nDistribución de d.time:")
display(
    day3_deaths["d.time"]
    .value_counts()
    .sort_index()
    .head(15)
)

print("\n=== VERIFICACIONES ===")
assert len(hospital_deaths) == 2360
assert len(day3_deaths) == 138

print("Auditoría completada correctamente.")


=== AUDITORÍA DE TIEMPO DE FALLECIMIENTO ===
Fallecidos hospitalarios: 2360

=== RESUMEN DE d.time ===


count    2360.000000
mean       17.469492
std        21.839309
min         3.000000
25%         6.000000
50%        10.000000
75%        20.000000
max       339.000000
Name: d.time, dtype: float64


=== TIEMPOS DE SEGUIMIENTO ===
d.time <= 1: 0
d.time <= 2: 0
d.time <= 3: 138

=== FALLECIDOS CON slos = 3 ===
Cantidad: 138

Distribución de d.time:


d.time
3    138
Name: count, dtype: int64


=== VERIFICACIONES ===
Auditoría completada correctamente.


In [30]:

# Resumen reproducible de la auditoría temporal
temporal_audit = pd.DataFrame([
    {
        "Indicador": "Total de pacientes",
        "Valor": len(df)
    },
    {
        "Indicador": "Pacientes con hday > 1",
        "Valor": int((df["hday"] > 1).sum())
    },
    {
        "Indicador": "Pacientes con slos < 3",
        "Valor": int((df["slos"] < 3).sum())
    },
    {
        "Indicador": "Pacientes con slos = 3",
        "Valor": int((df["slos"] == 3).sum())
    },
    {
        "Indicador": "Fallecidos hospitalarios",
        "Valor": int((df["hospdead"] == 1).sum())
    },
    {
        "Indicador": "Fallecidos con d.time < 3",
        "Valor": int(
            (
                (df["hospdead"] == 1) &
                (df["d.time"] < 3)
            ).sum()
        )
    },
    {
        "Indicador": "Fallecidos con slos = 3 y d.time = 3",
        "Valor": int(
            (
                (df["hospdead"] == 1) &
                (df["slos"] == 3) &
                (df["d.time"] == 3)
            ).sum()
        )
    }
])

# Guardar en results/tables
audit_path = RESULTS_DIR / "temporal_audit.csv"
temporal_audit.to_csv(audit_path, index=False)

print("=== AUDITORÍA TEMPORAL ===")
display(temporal_audit)

print("\n=== ARCHIVO GENERADO ===")
print(audit_path)

print("\n=== VERIFICACIONES ===")
assert audit_path.exists()
assert temporal_audit.loc[
    temporal_audit["Indicador"] ==
    "Fallecidos con slos = 3 y d.time = 3",
    "Valor"
].iloc[0] == 138

print("Auditoría temporal guardada correctamente.")


=== AUDITORÍA TEMPORAL ===


,Indicador,Valor
0,Total de pacientes,9105
1,Pacientes con hday > 1,3157
2,Pacientes con slos < 3,0
3,Pacientes con slos = 3,347
4,Fallecidos hospitalarios,2360
5,Fallecidos con d.time < 3,0
6,Fallecidos con slos = 3 y d.time = 3,138



=== ARCHIVO GENERADO ===
C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\results\tables\temporal_audit.csv

=== VERIFICACIONES ===
Auditoría temporal guardada correctamente.


In [31]:

instructions = """# P2 — Instrucciones para entrenamiento de modelos

## 1. Dataset y particiones

Dataset: SUPPORT2
Variable objetivo: hospdead

- Entrenamiento: 7284 pacientes
- Prueba: 1821 pacientes
- Características seleccionadas: 27
- División estratificada: 80/20
- Semilla: 42

Las particiones se encuentran en:

- dataset/processed/train_indices.csv
- dataset/processed/test_indices.csv

## 2. Cargar los datos

Ejecutar desde la raíz del repositorio:

```python
from preprocessing import load_data, build_preprocessor

X_train, X_test, y_train, y_test = load_data()
```

## 3. Construir un modelo

Para regresión logística:

```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", build_preprocessor(scaled=True)),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
```

Para árboles y Random Forest:

```python
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

model = Pipeline([
    ("preprocessor", build_preprocessor(scaled=False)),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)
```

## 4. Validación cruzada

Realizar la selección de modelos e hiperparámetros
exclusivamente sobre X_train e y_train.

Utilizar validación cruzada estratificada.

Mantener el preprocesador dentro del Pipeline,
para evitar fuga de información entre folds.

No utilizar X_test para seleccionar hiperparámetros.

## 5. Métricas

Reportar como mínimo:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Matriz de confusión

También se recomienda Average Precision,
por el desbalance entre las clases.

La clase positiva es hospdead = 1.

## 6. Baseline

Resultados de prueba del DummyClassifier(strategy="prior"):

- Accuracy: 0.7408
- Precision: 0.0000
- Recall: 0.0000
- F1-score: 0.0000
- ROC-AUC: 0.5000
- Average Precision: 0.2592

Archivo: results/tables/baseline_metrics.csv

## 7. Limitaciones metodológicas

El experimento utiliza características del dataset SUPPORT2
y se considera una clasificación retrospectiva
de mortalidad hospitalaria.

Las evaluaciones clínicas se relacionan con el tercer día
del estudio, que no necesariamente coincide con el
tercer día de hospitalización.

La información disponible no permite verificar las horas
exactas de las mediciones y de los desenlaces.

Se identificaron 138 pacientes fallecidos con
slos = 3 y d.time = 3.

Por ello, los resultados no deben interpretarse como
validación clínica prospectiva.

Consultar results/tables/temporal_audit.csv.
"""

# Guardar archivo Markdown
instructions_path = ROOT / "docs" / "P2_INSTRUCCIONES.md"

instructions_path.write_text(
    instructions,
    encoding="utf-8"
)

print("=== ARCHIVO GENERADO ===")
print(instructions_path)

print("\n=== VERIFICACIONES ===")
print("Archivo existe:", instructions_path.exists())

print(
    "Tamaño en bytes:",
    instructions_path.stat().st_size
)

assert instructions_path.exists()
assert "load_data" in instructions_path.read_text(
    encoding="utf-8"
)
assert "build_preprocessor" in instructions_path.read_text(
    encoding="utf-8"
)

print("Instrucciones de P2 guardadas correctamente.")


=== ARCHIVO GENERADO ===


C:\Users\gihua\Desktop\0. UTEC\1. MACHINE LEARNING\proyecto-test\Prediccion_mortalidad_hospitalaria_ml\docs\P2_INSTRUCCIONES.md

=== VERIFICACIONES ===


Archivo existe: True
Tamaño en bytes: 2734
Instrucciones de P2 guardadas correctamente.


In [32]:

# Localizar requirements.txt
requirements_path = ROOT / "requirements.txt"

print("=== DEPENDENCIAS ACTUALES ===")

requirements = requirements_path.read_text(
    encoding="utf-8"
)

print(requirements)

# Incorporar scikit-learn si todavía no existe
installed_packages = [
    line.strip().split("==")[0].lower()
    for line in requirements.splitlines()
    if line.strip() and not line.strip().startswith("#")
]

if "scikit-learn" not in installed_packages:
    requirements = requirements.rstrip() + "\nscikit-learn==1.9.1\n"

    requirements_path.write_text(
        requirements,
        encoding="utf-8"
    )

print("\n=== DEPENDENCIAS ACTUALIZADAS ===")

updated_requirements = requirements_path.read_text(
    encoding="utf-8"
)

print(updated_requirements)

print("\n=== VERIFICACIONES ===")

assert "scikit-learn" in updated_requirements
assert "pandas" in updated_requirements
assert "numpy" in updated_requirements
assert "matplotlib" in updated_requirements

print("requirements.txt actualizado correctamente.")

=== DEPENDENCIAS ACTUALES ===
pandas==2.2.3
numpy==2.1.3
matplotlib==3.10.0
scikit-learn==1.9.1
scipy==1.17.1
joblib==1.6.0

# Ejecución de notebooks y pruebas (P2)
ipykernel==7.3.0
nbformat==5.11.1
nbclient==0.11.0
pytest==9.2.0


=== DEPENDENCIAS ACTUALIZADAS ===
pandas==2.2.3
numpy==2.1.3
matplotlib==3.10.0
scikit-learn==1.9.1
scipy==1.17.1
joblib==1.6.0

# Ejecución de notebooks y pruebas (P2)
ipykernel==7.3.0
nbformat==5.11.1
nbclient==0.11.0
pytest==9.2.0


=== VERIFICACIONES ===
requirements.txt actualizado correctamente.
